# 04 - Metabelian twisted signature functions

This notebook follows the complete implemented route from a character on a common-`p` iterated torus knot to a coverage-aware jump profile and, when the hypotheses permit it, a normalized averaged twisted signature function.

The route combines three pieces:

1. Yanagida's explicit torus-knot matrices for the outer pattern;
2. the character-orbit basis conversion and ordinary companion signature; and
3. the divisible-winding branch of the satellite formula in BCP-II, Theorem 4.19.

The notebook also demonstrates an unresolved exceptional root. That expected failure is part of the mathematical result: an unavailable local pairing is reported, never replaced by a guessed zero.

## 1. Setup

In [ ]:
from pathlib import Path
import sys

repository_root = Path.cwd()
if repository_root.name == "notebooks":
    repository_root = repository_root.parent
source_directory = repository_root / "src"
if str(source_directory) not in sys.path:
    sys.path.insert(0, str(source_directory))

from sage.all import QQ
from gaknot import (
    BranchedCoverHomology,
    Character,
    GeneralizedAlgebraicKnot,
    YanagidaTorusData,
    iterated_torus_metabelian_signature_function,
    iterated_torus_metabelian_signature_jumps,
    yanagida_signature_profile,
)

## 2. Yanagida's global matrix data

For the torus pattern `T(m,n)` and orbit `b`, `YanagidaTorusData` constructs exact matrices over a rational-function field with cyclotomic coefficients. Here we use `T(2,5)` and the orbit `(1,4)`, whose entries sum to zero modulo five.

In [ ]:
data = YanagidaTorusData(2, 5, (1, 4))

print("normalized orbit b:", data.b)
print("canonical Bezout coefficients (r,s):", (data.r, data.s))
print("base ring:", data.function_field)
print("C =")
print(data.C)
print("X = C^n =")
print(data.X)
print("Y =")
print(data.Y)
print("M = X^s Y^r =")
print(data.M)

The matrices satisfy the defining relations `C^m=tI`, `X=C^n`, and `M=X^sY^r`. The orbit appears on the diagonal of `Y`. Keeping all entries exact is essential because later specialization and inertia calculations must distinguish a true zero from a small numerical approximation.

In [ ]:
identity = data.C.parent().one()
{
    "C^m = tI": data.C ** data.m == data.t * identity,
    "X = C^n": data.X == data.C ** data.n,
    "M = X^s Y^r": data.M == data.X ** data.s * data.Y ** data.r,
}

## 3. The local matrices `Theta_a` and `Psi_a`

A nonzero residue `a mod n` selects the root `t=zeta_n^a`. Equation (14) gives the local presentation `Theta_a` at every nontrivial root. Yanagida's Theorem 1.3 supplies the pairing matrix only when `a` is not congruent to any `-b_i`.

In [ ]:
generic_local = data.local_model(2)

print("a:", generic_local.a)
print("generic root:", generic_local.is_generic)
print("projection P_a =")
print(generic_local.projection)
print("Theta_a =")
print(generic_local.theta)

At `a=2`, neither orbit coordinate equals `-2 mod 5`, so the projection is the identity. The generic pairing object exposes `Theta_a` as its presentation and `t^n Psi` as the numerator matrix.

In [ ]:
generic_pairing = generic_local.pairing

print("presentation is Theta_a:",
      generic_pairing.presentation == generic_local.theta)
print("pairing matrix size:",
      (generic_pairing.matrix.nrows(), generic_pairing.matrix.ncols()))
print("symmetric local order:")
print(generic_pairing.symmetric_order)

At `a=4`, the coordinate `b_0=1` equals `-4 mod 5`. Equation (14) still defines `Theta_a`, but the pairing theorem excludes this root.

In [ ]:
exceptional_local = data.local_model(4)

print("generic root:", exceptional_local.is_generic)
print("exceptional projection P_a =")
print(exceptional_local.projection)
print("Theta_a is still available:")
print(exceptional_local.theta)

try:
    exceptional_local.pairing
except ValueError as error:
    print("pairing unavailable:", error)

Rejecting the pairing here prevents the formal rational matrix `Psi` from being mistaken for a theorem-backed local pairing outside its hypotheses.

## 4. A coverage-aware torus-pattern profile

`yanagida_signature_profile` computes Hodge signs at generic roots and separately records every exceptional root. In this `T(2,5)` example, each exceptional local module has dimension zero, so its jump is rigorously zero even though Theorem 1.3 does not apply there.

In [ ]:
pattern_profile = yanagida_signature_profile(data)

generic_summary = tuple(
    (jump.a, jump.argument, jump.module_dimension, jump.jump)
    for jump in pattern_profile.generic_jumps
)
exceptional_summary = tuple(
    (root.a, root.argument, root.module_dimension, root.jump_is_resolved)
    for root in pattern_profile.exceptional_roots
)

print("generic roots (a, argument, dimension, jump):", generic_summary)
print("exceptional roots (a, argument, dimension, resolved):",
      exceptional_summary)
print("all known nontrivial-root jumps:", pattern_profile.known_jump_values)
print("complete away from t=1:",
      pattern_profile.is_complete_at_nontrivial_roots)

A computed zero at a generic root and a zero forced by a zero-dimensional exceptional module are both known results. A positive-dimensional exceptional module, by contrast, remains unresolved.

## 5. End-to-end common-`p` satellite calculation

Now let the outer `T(2,5)` pattern act on the trefoil. The cable description runs from the inner companion outward. The character description follows homology layers in the reverse, outer-to-inner order.

In [ ]:
cable = GeneralizedAlgebraicKnot.iterated_torus_knot(
    [(2, 3), (2, 5)]
)
homology = BranchedCoverHomology(cable, 2)
character = Character(
    homology,
    [[[QQ(1) / 5], []]],
)

# Compute the normalized function once. Its jump_result property retains the
# coverage-aware intermediate object discussed in this section.
twisted_signature = iterated_torus_metabelian_signature_function(
    cable,
    character,
)
jump_result = twisted_signature.jump_result

print("cable sequence:", jump_result.cable_sequence)
print("deck orbit:", jump_result.orbit.a_values)
print("phase arguments:", jump_result.orbit.phase_arguments)
print("Theorem 4.19 branch:", jump_result.satellite_result.case)
print("known total jumps:", jump_result.total_profile.known_jumps)
print("unresolved arguments:", jump_result.unresolved_arguments)

Because the winding and cover degree are both two, Theorem 4.19 uses the `ordinary_companion` branch. It adds two phase-shifted copies of the trefoil's ordinary signature to the Yanagida pattern profile. The six listed nontrivial-root jumps are proved; the only original gap is `t=1`, represented by argument zero.

In [ ]:
for index, summand in enumerate(jump_result.companion_summands):
    print(f"companion summand {index}:", summand.known_jumps)

The diagnostic result remains intentionally incomplete at this stage. Calling `jump_at(0)` would raise `NotImplementedError`, making it impossible to confuse known nontrivial-root data with a globally normalized function.

## 6. Normalize the averaged twisted signature

For this nontrivial character of prime-power order, BCP-II, Theorem 4.14 gives representability and the normalization at `t=1`. Since no nontrivial root is unresolved, the single root-one jump can be inferred from the zero-total-jump condition.

In [ ]:
print("character order:", twisted_signature.character_order)
print("root-one jump inferred:",
      twisted_signature.signature_function.root_one_jump_inferred)
print("complete jump profile:", twisted_signature.jump_profile.known_jumps)

samples = [QQ(0), QQ(1) / 10, QQ(1) / 2, QQ(9) / 10]
print("normalized signature values:")
print({x: twisted_signature(x) for x in samples})
print("Casson--Gordon difference at 1/10:",
      twisted_signature.casson_gordon_signature_difference_at(QQ(1) / 10))

The final method returns `sign_av_omega(tau)-sign_av_1(tau)`, which is the negative of the normalized averaged twisted signature value under the convention used by Theorem 4.14(b). The original `jump_result` is retained inside the function result, so the use of representability to fill the root-one gap remains auditable.

## 7. A genuine exceptional-root coverage gap

For the outer pattern `T(3,4)` and the displayed order-four character, the roots at arguments `1/4` and `3/4` carry positive-dimensional exceptional modules. Representability determines only the **sum** of missing jumps; it cannot determine the two local jumps separately.

In [ ]:
exceptional_cable = GeneralizedAlgebraicKnot.iterated_torus_knot(
    [(3, 2), (3, 4)]
)
exceptional_character = Character(
    BranchedCoverHomology(exceptional_cable, 3),
    [[[QQ(1) / 4, 0], []]],
)

partial = iterated_torus_metabelian_signature_jumps(
    exceptional_cable,
    exceptional_character,
)

print("orbit:", partial.orbit.a_values)
print("unresolved arguments:", partial.unresolved_arguments)
print("exceptional dimensions:", tuple(
    (root.a, root.module_dimension)
    for root in partial.yanagida_profile.unresolved_exceptional_roots
))

nontrivial_gaps = tuple(
    argument for argument in partial.unresolved_arguments if argument != 0
)
print("complete normalization available:", not nontrivial_gaps)

This is not a software crash or a numerical failure. It is a precise statement about current theorem coverage. The partial object can still be inspected, but no complete signature function is manufactured. Calling `iterated_torus_metabelian_signature_function` with this input raises `NotImplementedError`; the pytest suite checks that contract directly.

## 8. Current scope and references

The high-level route currently supports one positive iterated torus knot whose cabling layers share a common first parameter `p`, with a character on the `p`-fold cover. Connected sums, negative summands, nonconstant winding parameters, and surviving exceptional local modules require further implementation or further mathematical input.

The main references for this notebook are:

- Koki Yanagida, *Blanchfield pairings and twisted Blanchfield pairings of torus knots*, [arXiv:2602.07575v2](https://arxiv.org/abs/2602.07575v2);
- Borodzik--Conway--Politarczyk, *Twisted Blanchfield pairings and twisted signatures II*, [arXiv:1809.08791](https://arxiv.org/abs/1809.08791), especially Theorems 4.14 and 4.19; and
- Conway--Kim--Politarczyk, *Non-slice linear combinations of iterated torus knots*, [arXiv:1910.01368](https://arxiv.org/abs/1910.01368), for the character-orbit conventions.

## Exercises

1. Replace the outer pattern by `T(2,7)` and compute the generic and exceptional summaries.
2. Build the three-layer knot `T(2,3; 2,5; 2,7)` with the outer `1/7` character and inspect which lower layers form the ordinary companion.
3. For the exceptional `T(3,4)` example, list the known jumps separately from the unresolved roots and explain why a zero-total-jump equation is insufficient to recover both missing values.